# Example queries: `report` (resstock_oedi)

Auto-generated from `tests/query_snapshots/report.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading resstock_2024_amy2018_release_2 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `report_buildings_by_change_ok`

report.get_buildings_by_change for upgrade 1, change_type='ok-chng' — buildings whose upgrade succeeded with an acceptable energy change. Pins the change-condition SQL shape now that the OEDI applicability fix lets the report layer resolve.


In [3]:
result = bsq.report.get_buildings_by_change(upgrade_id='1', change_type='ok-chng')
result.head() if hasattr(result, 'head') else result


[264875,
 264963,
 265162,
 265209,
 265252,
 265261,
 265304,
 265345,
 265351,
 265403,
 265409,
 265444,
 265476,
 265487,
 265496,
 265528,
 265543,
 265556,
 265586,
 265638,
 265655,
 265698,
 265737,
 265739,
 265750,
 265797,
 265806,
 265884,
 265890,
 265933,
 265942,
 265948,
 265989,
 266000,
 266032,
 266073,
 266084,
 266131,
 266144,
 266196,
 266218,
 266235,
 266246,
 266267,
 266278,
 266295,
 266330,
 266347,
 266414,
 266429,
 266446,
 266513,
 266530,
 266545,
 266565,
 266597,
 266612,
 266629,
 266632,
 266664,
 266696,
 266724,
 266780,
 266791,
 266828,
 266832,
 266862,
 266879,
 266899,
 266916,
 266963,
 267015,
 267026,
 267078,
 267110,
 267177,
 267220,
 267229,
 267261,
 267319,
 267371,
 267403,
 267444,
 267472,
 267554,
 267614,
 267623,
 267653,
 267701,
 267707,
 267765,
 267800,
 267858,
 267901,
 267948,
 267968,
 268000,
 268035,
 268112,
 268147,
 268246,
 268302,
 268345,
 268362,
 268386,
 268397,
 268414,
 268429,
 268481,
 268498,
 268513,
 

## `report_successful_simulation_count_co`

report.get_successful_simulation_count restricted to CO. Pins the metadata-only count shape. Method was broken before fix in 2026-04-26: the call passed `bs_only=True` to _add_restrict() which doesn't accept that kwarg, causing TypeError on every call with a non-empty restrict.


In [4]:
result = bsq.report.get_successful_simulation_count(restrict=[('state', ['CO'])])
result.head() if hasattr(result, 'head') else result


,count
0,9425


## `report_successful_simulation_count_no_restrict`

report.get_successful_simulation_count with no restrict — pins the no-WHERE branch (only the implicit applicability=true filter). Pairs with report_successful_simulation_count_co to cover both restrict and no-restrict paths.


In [5]:
result = bsq.report.get_successful_simulation_count()
result.head() if hasattr(result, 'head') else result


,count
0,549718


## `report_success_report`

report.get_success_report — returns three SQL statements (baseline counts, per-upgrade counts, change-type breakdown). Joined into one snapshot via the harness's MULTI_SQL_SEPARATOR. Pins the most-used reporting entry point. Requires the OEDI applicability fix in _rename_completed_status_column (commit added a lowercase-string normalize so True/False booleans map correctly).


In [6]:
# NOTE: cell intentionally not executed — `report.get_success_report` issues
# additional Athena queries beyond the snapshot test cache.
# Uncomment to run live (incurs scan cost):
# bsq.report.get_success_report(trim_missing_bs='auto')


## `report_check_ts_bs_integrity`

report.check_ts_bs_integrity — every BuildStockQuery(skip_reports=False) init runs this. Pins the SQL shape of all queries it submits: per-upgrade ts distinct counts, metadata distinct counts (baseline + per-upgrade), duplicate-row checks, rows-per-building. Marked nondeterministic to skip data check — _get_rows_per_building scans the full TS table and would be a multi-TB landmine to actually run. SQL-shape regression coverage for the comma-join class of bug fixed in commit 9cd148d (md_table column refs mixed with bs_table predicates produced FROM x, x AS bs Cartesians that Athena rejected with 'mismatched input GROUP').


In [7]:
# NOTE: cell intentionally not executed — `report.check_ts_bs_integrity` issues
# additional Athena queries beyond the snapshot test cache.
# Uncomment to run live (incurs scan cost):
# bsq.report.check_ts_bs_integrity()
